<a href="https://colab.research.google.com/github/raiyashu2004/Achintya-Rai/blob/raiyashu2004-dailyquestion/summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 — Runtime check: make sure you selected GPU (Runtime -> Change runtime type -> GPU)
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))


PyTorch version: 2.8.0+cu126
CUDA available: True
Device: cuda


In [2]:
# Cell 2 — Install required packages (run once)
# Hugging Face Transformers, datasets, accelerate, sentencepiece, and evaluate
!pip install -q transformers datasets accelerate sentencepiece evaluate huggingface_hub
# Optional: for progress bars and nicer logging
!pip install -q tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00


In [3]:
# Cell 3 — Imports and basic config
import os
import math
import random
from dataclasses import dataclass, field
from typing import Optional, Dict

import numpy as np
import torch
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    pipeline,
    set_seed,
)
set_seed(42)

In [4]:
# Cell 4 — Select model and tokenizer
MODEL_NAME = "facebook/bart-large-cnn"  # good general summarization checkpoint
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to("cuda" if torch.cuda.is_available() else "cpu")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [5]:
# Load your CSV datasets
from datasets import load_dataset

raw_datasets = load_dataset(
    "csv",
    data_files={
        "train": "train.csv",
        "validation": "test.csv"   # using test.csv as validation
    }
)

print(raw_datasets)
small_train = raw_datasets["train"]
small_val   = raw_datasets["validation"]


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'summary'],
        num_rows: 80
    })
    validation: Dataset({
        features: ['text', 'label', 'summary'],
        num_rows: 20
    })
})


In [6]:
max_input_length = 1024
max_target_length = 128

def preprocess_function(examples):
    inputs = examples["text"]       # your dataset's input column
    targets = examples["summary"]   # your dataset's summary column

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=max_target_length,
            truncation=True,
            padding="max_length"
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = small_train.map(
    preprocess_function,
    batched=True,
    remove_columns=small_train.column_names
)
tokenized_val = small_val.map(
    preprocess_function,
    batched=True,
    remove_columns=small_val.column_names
)


Map:   0%|          | 0/80 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4006: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [7]:
!pip install rouge_score


  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=3aa0fb8557d733d813be6f3adb3c71d1af94a7c8ad9d88273913fd089b6977cc
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [8]:
# Cell 7 — Data collator and metrics
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=tokenizer.pad_token_id)

# Install missing dependency if not already installed
try:
    import rouge_score
except ImportError:
    !pip install -q rouge_score

import evaluate
rouge = evaluate.load("rouge")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [lab.strip() for lab in labels]
    return preds, labels

def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {k: round(v * 100, 4) for k, v in result.items()}
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)
    return result


In [9]:
# Cell 8 — Training arguments (Colab-friendly)
output_dir = "./bart-summarizer-cnn-small"
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",   # new name
    per_device_train_batch_size=2,   # small for Colab GPU memory
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,   # emulate larger batch by accumulation
    learning_rate=3e-5,
    num_train_epochs=1,              # set to 3-4 for real training
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),  # enable mixed precision if GPU supports it
    logging_steps=50,
    save_strategy="epoch",
    push_to_hub=False,               # set True if you want to push to HF hub
    report_to="none",                # disable wandb or other reporting unless configured
)

In [10]:
# Cell 9 — Trainer setup
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


/tmp/ipython-input-1468050422.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,No log,8.035492,37.490400,35.481000,37.522800,37.518700,61.650000


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3922: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=5, training_loss=9.706956481933593, metrics={'train_runtime': 82.6634, 'train_samples_per_second': 0.968, 'train_steps_per_second': 0.06, 'total_flos': 173368368168960.0, 'train_loss': 9.706956481933593, 'epoch': 1.0})

In [12]:
# Cell 11 — Save fine-tuned model and tokenizer
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("Model and tokenizer saved to:", output_dir)

Model and tokenizer saved to: ./bart-summarizer-cnn-small


In [13]:
# Cell 12 — Reload the fine-tuned model
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

inferred_tokenizer = AutoTokenizer.from_pretrained(output_dir)
inferred_model = AutoModelForSeq2SeqLM.from_pretrained(output_dir).to(
    "cuda" if torch.cuda.is_available() else "cpu"
)


/usr/local/lib/python3.12/dist-packages/transformers/models/bart/configuration_bart.py:177: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(


In [14]:
# Cell 13 — Summarization pipeline
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model=inferred_model,
    tokenizer=inferred_tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

example_text = """
Artificial Intelligence is transforming industries rapidly.
It has applications in healthcare, finance, education, and transportation.
However, ethical concerns around bias and job displacement remain critical.
"""

summary = summarizer(example_text, max_length=80, min_length=20, do_sample=False)[0]["summary_text"]
print("Summary:", summary)


Device set to use cuda:0
Your max_length is set to 80, but your input_length is only 42. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=21)


Summary: Artificial Intelligence is transforming industries rapidly. It has applications in healthcare, finance, education, and transportation. ethical concerns around bias and job displacement remain critical.
